# Import the libraries and configure a spark session

In [1]:
from pyspark.sql import SparkSession
import os
import numpy as np
import pandas as pd
from pyspark.sql.functions import pandas_udf
import pyspark.sql.functions as sf


In [2]:
# Get the access and secret keys to connect to s3
access_key = os.getenv("S3_ACCESS_KEY")
secret_key = os.getenv("S3_SECRET_KEY") 

# Create a spark session including modules to connect to s3
spark = SparkSession.builder \
    .master("spark://master:7077")\
    .appName("Anomaly Detection")\
    .config('spark.jars.packages', 'org.apache.hadoop:hadoop-aws:3.4.1,org.apache.hadoop:hadoop-common:3.4.1')\
    .config("spark.sql.execution.arrow.pyspark.enabled", "true")\
    .config("spark.sql.execution.arrow.pyspark.fallback.enabled", "false")\
    .config('spark.hadoop.fs.s3a.aws.credentials.proviAnvoder', 'org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider')\
    .config('spark.hadoop.fs.s3a.access.key', access_key)\
    .config('spark.hadoop.fs.s3a.secret.key', secret_key)\
    .config('spark.hadoop.fs.s3a.endpoint', 'https://cloud-areapd.pd.infn.it:5210')\
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.metadatastore.impl", "org.apache.hadoop.fs.s3a.s3guard.NullMetadataStore") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled","false") \
    .config("com.amazonaws.sdk.disableCertChecking","true") \
    .getOrCreate()

:: loading settings :: url = jar:file:/usr/local/spark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/pandreac/.ivy2.5.2/cache
The jars for the packages stored in: /home/pandreac/.ivy2.5.2/jars
org.apache.hadoop#hadoop-aws added as a dependency
org.apache.hadoop#hadoop-common added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-82b826a0-a34a-472e-9725-0c02a4654b6a;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.4.1 in central
	found software.amazon.awssdk#bundle;2.24.6 in central
	found org.wildfly.openssl#wildfly-openssl;1.1.3.Final in central
	found org.apache.hadoop#hadoop-common;3.4.1 in central
	found org.apache.hadoop.thirdparty#hadoop-shaded-protobuf_3_25;1.3.0 in central
	found org.apache.hadoop#hadoop-annotations;3.4.1 in central
	found org.apache.hadoop.thirdparty#hadoop-shaded-guava;1.3.0 in central
	found com.google.guava#guava;27.0-jre in central
	found com.google.guava#failure

# Task 1: conversion of alarms

First of all, we will convert the "A5" and "A9" variables from their integer representation to their bit string one. This will help identifying the required alarms. \
First, we have to read the data file from the CloudVeneto "bucket", converted into the *Parquet* format by  `df_spark.write.mode("overwrite").parquet("s3a://MAPDB-Group5/data_parquet")`

In [3]:
# Read data from the CloudVeneto "bucket" through s3
df_spark = spark.read.parquet("s3a://MAPDB-Group5/data_parquet")

In [4]:
# Converting the A5 and A9 metrics to their bit-string representation
# Note: all the other values are converted into strings because Spark
# wants the same type returned by when/otherwise 
df_spark = df_spark.withColumn(
    "BitString",
    sf.when(
        (df_spark.metric == 'A5') | (df_spark.metric == 'A9'),
        sf.lpad(sf.bin(df_spark.value), 16, '0')
    ).otherwise(df_spark.value.cast('string'))
)

# After the conversion is done, we can persist the data
df_spark = df_spark.persist()
df_spark.count()

142102770

Now that we have converted the 'A5' and 'A9' variables to their bit-string representation, we can identify the requested alarms by checking if 1+ bit(s) in position 6, 7 and 8 (staring from the LSB), are 1 in either or both of them, which means that engines are overheating.

In [ ]:
df_spark = df_spark.withColumn(
    "is_overheated",
    sf.when(
        sf.substring(df_spark.BitString, 6, 3) != '000',
        True
    )\
    .otherwise(False)
)